In [ ]:
!pip install pandas requests beautifulsoup4


In [ ]:
import os
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup

# Function to create a requests session with retries
def create_session(retries=5, backoff_factor=1, status_forcelist=(500, 502, 503, 504)):
    session = requests.Session()
    retry = Retry(
        total=retries,
        read=retries,
        connect=retries,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

# Create a session with retry logic
session = create_session()

# Path to your CSV file containing paper metadata, including a "FullTextURL" column (adjust path if needed)
csv_path = "result.csv"
df = pd.read_csv(csv_path)

# Ensure that the CSV has the required column
if 'FullTextURL' not in df.columns:
    raise ValueError("CSV file must contain a 'FullTextURL' column.")

# Prepare output directories for downloaded PDFs and extracted texts
pdf_dir = "downloaded_pdfs"
text_dir = "extracted_texts"
os.makedirs(pdf_dir, exist_ok=True)
os.makedirs(text_dir, exist_ok=True)

# Loop over each row and attempt to download the PDF; if that fails, extract text from the page
for idx, row in df.iterrows():
    url = row.get('FullTextURL')
    if pd.isna(url) or not url:
        print(f"Row {idx}: No FullTextURL available, skipping.")
        continue

    # Use the title if available for filename; otherwise, default to paper_{idx}
    title = row.get('title', f'paper_{idx}')
    # Clean title for filename (allow alphanumerics, space, underscore, hyphen)
    safe_title = "".join(x for x in str(title) if x.isalnum() or x in " _-")

    try:
        print(f"Attempting to download content for row {idx+1} from: {url}")
        response = session.get(url, timeout=20)
        if response.status_code == 200:
            content_type = response.headers.get('Content-Type', '').lower()
            # If the URL indicates a PDF, or the content type contains 'pdf', then save as PDF
            if "pdf" in content_type or url.lower().endswith(".pdf"):
                filename = f"{safe_title}.pdf"
                filepath = os.path.join(pdf_dir, filename)
                with open(filepath, "wb") as f:
                    f.write(response.content)
                print(f"Saved PDF as: {filepath}")
            else:
                # Otherwise, assume HTML and try to extract text
                soup = BeautifulSoup(response.content, "html.parser")
                page_text = soup.get_text(separator="\n", strip=True)
                if page_text:
                    filename = f"{safe_title}.txt"
                    filepath = os.path.join(text_dir, filename)
                    with open(filepath, "w", encoding="utf-8") as f:
                        f.write(page_text)
                    print(f"Extracted and saved text as: {filepath}")
                else:
                    print(f"No text extracted from: {url}")
        else:
            print(f"Failed to download {url}: Status code {response.status_code}")
    except Exception as e:
        print(f"Error downloading {url}: {e}")


Attempting to download content for row 1 from: https://par.nsf.gov/servlets/purl/10179541
Saved PDF as: downloaded_pdfs/paper_0.pdf
Attempting to download content for row 2 from: http://www.geo.utexas.edu/courses/387H/PAPERS/Oppenheimer%201998%20Nature.pdf
Saved PDF as: downloaded_pdfs/paper_1.pdf
Attempting to download content for row 3 from: http://www.uib.no/sites/w3.uib.no/files/attachments/joughinnatgeoreview2011.pdf
Saved PDF as: downloaded_pdfs/paper_2.pdf
Attempting to download content for row 4 from: https://www.science.org/doi/full/10.1126/science.aaz5487
Failed to download https://www.science.org/doi/full/10.1126/science.aaz5487: Status code 403
Attempting to download content for row 5 from: https://hal.science/hal-04890701/document
Saved PDF as: downloaded_pdfs/paper_4.pdf
Attempting to download content for row 6 from: https://core.ac.uk/download/pdf/578111877.pdf
Saved PDF as: downloaded_pdfs/paper_5.pdf
Row 6: No FullTextURL available, skipping.
Row 7: No FullTextURL avai

Error downloading https://www.scirp.org/journal/paperinformation?paperid=106571: HTTPSConnectionPool(host='www.scirp.org', port=443): Max retries exceeded with url: /journal/paperinformation?paperid=106571 (Caused by ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')))
Attempting to download content for row 164 from: https://www.jstage.jst.go.jp/article/pjab/89/7/89_PJA8907B-01/_pdf
Saved PDF as: downloaded_pdfs/paper_163.pdf
Attempting to download content for row 165 from: https://scholarworks.uark.edu/cgi/viewcontent.cgi?article=2183&context=jaas
Saved PDF as: downloaded_pdfs/paper_164.pdf
Attempting to download content for row 166 from: https://www.academia.edu/download/45059021/Concluding_Remarks_Recent_Changes_in_Ant20160425-14762-qst2lr.pdf
Failed to download https://www.academia.edu/download/45059021/Concluding_Remarks_Recent_Changes_in_Ant20160425-14762-qst2lr.pdf: Status code 404
Attempting to download content for row 167 f

Error downloading https://www.scirp.org/journal/paperinformation?paperid=131458: HTTPSConnectionPool(host='www.scirp.org', port=443): Max retries exceeded with url: /journal/paperinformation?paperid=131458 (Caused by ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')))
Attempting to download content for row 181 from: https://pubs.rsc.org/en/content/articlehtml/2022/em/d2em00273f


Error downloading https://pubs.rsc.org/en/content/articlehtml/2022/em/d2em00273f: HTTPSConnectionPool(host='pubs.rsc.org', port=443): Max retries exceeded with url: /en/content/articlehtml/2022/em/d2em00273f (Caused by ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')))
Attempting to download content for row 182 from: https://pordlabs.ucsd.edu/jsprintall/pub_dir/MA05CH14-Smith%5B001-019%5D.pdf
Saved PDF as: downloaded_pdfs/paper_181.pdf
Attempting to download content for row 183 from: https://www.researchgate.net/profile/Charles-Greene-5/publication/23679039_Arctic_climate_change_and_its_impacts_on_the_ecology_of_the_North_Atlantic/links/5b9d7ada92851ca9ed0d5ec0/Arctic-climate-change-and-its-impacts-on-the-ecology-of-the-North-Atlantic.pdf
Failed to download https://www.researchgate.net/profile/Charles-Greene-5/publication/23679039_Arctic_climate_change_and_its_impacts_on_the_ecology_of_the_North_Atlantic/links/5b9d7ada92851ca9ed0d

Error downloading https://www.scirp.org/html/7-2171554_107789.htm: HTTPSConnectionPool(host='www.scirp.org', port=443): Max retries exceeded with url: /html/7-2171554_107789.htm (Caused by ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')))
Attempting to download content for row 223 from: https://link.springer.com/article/10.1007/s00300-022-03059-8
Extracted and saved text as: extracted_texts/paper_222.txt
Attempting to download content for row 224 from: https://www.sciencedirect.com/science/article/pii/S1873965218301543
Failed to download https://www.sciencedirect.com/science/article/pii/S1873965218301543: Status code 403
Attempting to download content for row 225 from: https://www.researchgate.net/profile/John-Mcbride-4/publication/241100083_Climate_change_impacts_on_tropical_cyclones_and_extreme_sea_levels_in_the_South_Pacific_-_A_regional_assessment/links/5c1e3868a6fdccfc70614bc7/Climate-change-impacts-on-tropical-cyclones-and-

Extracted and saved text as: extracted_texts/paper_274.txt
Row 275: No FullTextURL available, skipping.
Row 276: No FullTextURL available, skipping.
Attempting to download content for row 278 from: https://egusphere.copernicus.org/preprints/2023/egusphere-2023-1753/egusphere-2023-1753.pdf
Saved PDF as: downloaded_pdfs/paper_277.pdf
Attempting to download content for row 279 from: https://cyberleninka.ru/article/n/black-carbon-as-a-factor-in-deglaciation-in-polar-and-mountain-ecosystems-a-review
Extracted and saved text as: extracted_texts/paper_278.txt
Attempting to download content for row 280 from: https://agupubs.onlinelibrary.wiley.com/doi/pdf/10.1029/2011RG000361
Failed to download https://agupubs.onlinelibrary.wiley.com/doi/pdf/10.1029/2011RG000361: Status code 403
Attempting to download content for row 281 from: https://royalsocietypublishing.org/doi/pdf/10.1098/rspa.2019.0458?download=true
Failed to download https://royalsocietypublishing.org/doi/pdf/10.1098/rspa.2019.0458?down

In [ ]:
pip install PyPDF2 pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.1 MB/s eta 0:00:00


In [ ]:
import os
import PyPDF2
import pandas as pd

# Directories for the two sources
pdf_folder = "downloaded_pdfs"
text_folder = "extracted_texts"

# Lists to store data from each source
data = []

# Function to extract text from a PDF using PyPDF2
def extract_text_from_pdf(filepath):
    text = ""
    try:
        with open(filepath, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print(f"Error processing {filepath}: {e}")
    return text.strip()

# Process PDFs: extract text from each PDF file in pdf_folder
for filename in os.listdir(pdf_folder):
    if filename.lower().endswith(".pdf"):
        filepath = os.path.join(pdf_folder, filename)
        print(f"Extracting text from PDF: {filename}")
        pdf_text = extract_text_from_pdf(filepath)
        data.append({
            "filename": filename,
            "text": pdf_text,
            "source": "PDF"
        })

# Process extracted text files: simply read their content
for filename in os.listdir(text_folder):
    if filename.lower().endswith(".txt"):
        filepath = os.path.join(text_folder, filename)
        print(f"Reading extracted text file: {filename}")
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                text_content = f.read()
            data.append({
                "filename": filename,
                "text": text_content.strip(),
                "source": "HTML-text"
            })
        except Exception as e:
            print(f"Error reading {filepath}: {e}")

# Create a DataFrame from the combined data
df = pd.DataFrame(data)

# Optionally preview the dataset
print(df.head())

# Save the dataset to a CSV file
output_csv = "combined_text_dataset.csv"
df.to_csv(output_csv, index=False)
print(f"Combined dataset saved as '{output_csv}'")


Extracting text from PDF: paper_69.pdf
Extracting text from PDF: paper_304.pdf
Extracting text from PDF: paper_19.pdf
Extracting text from PDF: paper_345.pdf
Extracting text from PDF: paper_78.pdf
Extracting text from PDF: paper_191.pdf
Extracting text from PDF: paper_13.pdf
Extracting text from PDF: paper_166.pdf
Error processing downloaded_pdfs/paper_166.pdf: EOF marker not found
Extracting text from PDF: paper_228.pdf
Extracting text from PDF: paper_190.pdf
Extracting text from PDF: paper_157.pdf


Extracting text from PDF: paper_270.pdf
Error processing downloaded_pdfs/paper_270.pdf: EOF marker not found
Extracting text from PDF: paper_79.pdf
Extracting text from PDF: paper_64.pdf
Extracting text from PDF: paper_72.pdf
Error processing downloaded_pdfs/paper_72.pdf: EOF marker not found
Extracting text from PDF: paper_125.pdf
Extracting text from PDF: paper_327.pdf
Extracting text from PDF: paper_8.pdf
Error processing downloaded_pdfs/paper_8.pdf: EOF marker not found
Extracting text from PDF: paper_112.pdf
Extracting text from PDF: paper_220.pdf
Error processing downloaded_pdfs/paper_220.pdf: EOF marker not found
Extracting text from PDF: paper_96.pdf
Error processing downloaded_pdfs/paper_96.pdf: EOF marker not found
Extracting text from PDF: paper_339.pdf
Extracting text from PDF: paper_269.pdf
Extracting text from PDF: paper_326.pdf
Error processing downloaded_pdfs/paper_326.pdf: EOF marker not found
Extracting text from PDF: paper_144.pdf
Error processing downloaded_pdfs/pap

[0, IndirectObject(285, 0, 132883103556304)]
[0, IndirectObject(281, 0, 132883103556304)]
[0, IndirectObject(277, 0, 132883103556304)]
[0, IndirectObject(255, 0, 132883103556304)]
[0, IndirectObject(285, 0, 132883103556304)]
[0, IndirectObject(298, 0, 132883103556304)]
[0, IndirectObject(294, 0, 132883103556304)]
[0, IndirectObject(290, 0, 132883103556304)]
[0, IndirectObject(285, 0, 132883103556304)]
[0, IndirectObject(285, 0, 132883103556304)]
[0, IndirectObject(302, 0, 132883103556304)]
[0, IndirectObject(298, 0, 132883103556304)]
[0, IndirectObject(290, 0, 132883103556304)]
[0, IndirectObject(302, 0, 132883103556304)]
[0, IndirectObject(298, 0, 132883103556304)]


Extracting text from PDF: paper_58.pdf
Error processing downloaded_pdfs/paper_58.pdf: EOF marker not found
Extracting text from PDF: paper_333.pdf


[0, IndirectObject(298, 0, 132883103556304)]
[0, IndirectObject(290, 0, 132883103556304)]
[0, IndirectObject(285, 0, 132883103556304)]
[0, IndirectObject(298, 0, 132883103556304)]
[0, IndirectObject(302, 0, 132883103556304)]
[0, IndirectObject(285, 0, 132883103556304)]
[0, IndirectObject(306, 0, 132883103556304)]
[0, IndirectObject(294, 0, 132883103556304)]
[0, IndirectObject(298, 0, 132883103556304)]
[0, IndirectObject(285, 0, 132883103556304)]
[0, IndirectObject(298, 0, 132883103556304)]
[0, IndirectObject(285, 0, 132883103556304)]
[0, IndirectObject(298, 0, 132883103556304)]
[0, IndirectObject(310, 0, 132883103556304)]
[0, IndirectObject(298, 0, 132883103556304)]
[0, IndirectObject(314, 0, 132883103556304)]
[0, IndirectObject(306, 0, 132883103556304)]
[0, IndirectObject(298, 0, 132883103556304)]
[0, IndirectObject(306, 0, 132883103556304)]
[0, IndirectObject(302, 0, 132883103556304)]
[0, IndirectObject(318, 0, 132883103556304)]
[0, IndirectObject(310, 0, 132883103556304)]
[0, Indire

Extracting text from PDF: paper_109.pdf
Extracting text from PDF: paper_246.pdf
Extracting text from PDF: paper_172.pdf
Extracting text from PDF: paper_364.pdf
Extracting text from PDF: paper_127.pdf
Error processing downloaded_pdfs/paper_127.pdf: EOF marker not found
Extracting text from PDF: paper_39.pdf
Extracting text from PDF: paper_154.pdf
Error processing downloaded_pdfs/paper_154.pdf: EOF marker not found
Extracting text from PDF: paper_52.pdf
Extracting text from PDF: paper_29.pdf
Extracting text from PDF: paper_230.pdf
Error processing downloaded_pdfs/paper_230.pdf: EOF marker not found
Extracting text from PDF: paper_222.pdf
Error processing downloaded_pdfs/paper_222.pdf: EOF marker not found
Extracting text from PDF: paper_296.pdf
Extracting text from PDF: paper_181.pdf
Extracting text from PDF: paper_332.pdf
Extracting text from PDF: paper_132.pdf
Error processing downloaded_pdfs/paper_132.pdf: EOF marker not found
Extracting text from PDF: paper_290.pdf
Error processing d